# Abstracción de un consultorio médico

Este notebook modela un consultorio médico aplicando los pilares de la programación orientada a objetos.

## Objetivo

Representar las clases `Persona`, `Medico`, `Paciente` y `Cita`, sus responsabilidades y las relaciones entre ellas.

- `Persona` abstrae los datos comunes: nombre, apellido y edad.
- `Medico` agrega especialidad y consultorio, y puede recetar.
- `Paciente` agrega EPS y estado, y puede agendar o cancelar citas.
- `Cita` relaciona un paciente con un médico mediante una fecha y un estado.

La relación adicional necesaria es que una cita pertenece a un paciente y tiene un médico agendado. Un paciente puede tener varias citas y un médico puede atender varias citas.

In [ ]:
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass
from datetime import datetime


class Persona(ABC):
    """Superclase con la información común de las personas del consultorio."""

    def __init__(self, nombre: str, apellido: str, edad: int) -> None:
        if not nombre.strip() or not apellido.strip():
            raise ValueError("El nombre y el apellido son obligatorios")
        if edad < 0:
            raise ValueError("La edad no puede ser negativa")
        self.nombre = nombre.strip()
        self.apellido = apellido.strip()
        self.edad = edad

    @abstractmethod
    def rol(self) -> str:
        """Devuelve el rol de la persona; cada subclase lo define."""

    def nombre_completo(self) -> str:
        return f"{self.nombre} {self.apellido}"


@dataclass
class Cita:
    """Asociación entre un paciente y un médico."""

    __fecha_agendamiento: datetime
    __medico_agendado: Medico
    __paciente: Paciente
    __estado: str = "agendada"

    @property
    def estado(self) -> str:
        """Expone el estado sin permitir modificarlo directamente."""
        return self._estado

    def cancelar(self) -> None:
        if self._estado == "cancelada":
            raise ValueError("La cita ya está cancelada")
        self._estado = "cancelada"


class Medico(Persona):
    def __init__(
        self,
        nombre: str,
        apellido: str,
        edad: int,
        especialidad: str,
        consultorio: str,
    ) -> None:
        super().__init__(nombre, apellido, edad)
        self.especialidad = especialidad
        self.consultorio = consultorio

    def rol(self) -> str:
        return f"Médico especialista en {self.especialidad}"

    def recetar(self, paciente: Paciente, medicamento: str) -> str:
        if paciente.estado.lower() != "activo":
            raise ValueError("No se puede recetar a un paciente inactivo")
        return (
            f"Receta de {medicamento} para {paciente.nombre_completo()} "
            f"por el Dr. {self.nombre_completo()}"
        )


class Paciente(Persona):
    def __init__(
        self,
        nombre: str,
        apellido: str,
        edad: int,
        eps: str,
        estado: str = "activo",
    ) -> None:
        super().__init__(nombre, apellido, edad)
        self.eps = eps
        self.estado = estado
        self._citas: list[Cita] = []

    def rol(self) -> str:
        return f"Paciente afiliado a {self.eps}"

    @property
    def citas(self) -> tuple[Cita, ...]:
        """Devuelve las citas como tupla para proteger la colección interna."""
        return tuple(self._citas)

    def agendar_cita(self, medico: Medico, fecha: datetime) -> Cita:
        if self.estado.lower() != "activo":
            raise ValueError("Un paciente inactivo no puede agendar citas")
        cita = Cita(fecha, medico, self)
        self._citas.append(cita)
        return cita

    def cancelar_cita(self, cita: Cita) -> None:
        if cita not in self._citas:
            raise ValueError("La cita no pertenece a este paciente")
        cita.cancelar()


# Ejemplo de uso
medico = Medico(
    "Laura", "Gómez", 42, "Medicina interna", "Consultorio 204"
)
paciente = Paciente("Carlos", "Pérez", 35, "SaludTotal")
cita = paciente.agendar_cita(medico, datetime(2026, 9, 15, 10, 30))

print(medico.rol())
print(paciente.rol())
print(medico.recetar(paciente, "Acetaminofén"))
print(f"Cita: {cita.fecha_agendamiento:%d/%m/%Y %H:%M} - {cita.estado}")
paciente.cancelar_cita(cita)
print(f"Estado después de cancelar: {cita.estado}")

assert len(paciente.citas) == 1
assert cita.medico_agendado is medico
assert cita.paciente is paciente
assert cita.estado == "cancelada"

Médico especialista en Medicina interna
Paciente afiliado a SaludTotal
Receta de Acetaminofén para Carlos Pérez por el Dr. Laura Gómez
Cita: 15/09/2026 10:30 - agendada
Estado después de cancelar: cancelada


## Diagrama UML de clases

```plantuml
@startuml
abstract class Persona {
  - nombre: str
  - apellido: str
  - edad: int
  + nombre_completo(): str
  + rol(): str
}

class Medico {
  - especialidad: str
  - consultorio: str
  + rol(): str
  + recetar(paciente: Paciente, medicamento: str): str
}

class Paciente {
  - eps: str
  - estado: str
  - _citas: list[Cita]
  + rol(): str
  + agendar_cita(medico: Medico, fecha: datetime): Cita
  + cancelar_cita(cita: Cita): None
}

class Cita {
  - fecha_agendamiento: datetime
  - _estado: str
  + estado: str
  + cancelar(): None
}

Persona <|-- Medico
Persona <|-- Paciente
Paciente "1" *-- "0..*" Cita : agenda
Medico "1" -- "0..*" Cita : atiende
Medico ..> Paciente : receta
@enduml
```

### Relaciones

- `Medico` y `Paciente` heredan de `Persona`: es una relación **es-un**.
- Un `Paciente` puede tener cero o muchas `Cita`; cada cita pertenece a un paciente.
- Un `Medico` puede atender cero o muchas `Cita`; cada cita tiene un médico agendado.
- `Medico` depende de `Paciente` al utilizarlo como receptor de una receta.
- El estado interno de `Cita` y la colección de citas del paciente se protegen mediante una interfaz pública controlada.